In [1]:
import json
import hashlib
import networkx as nx
from tqdm import tqdm


# ============================================================
# Hard-coded hyperparams (edit in notebook as needed)
# ============================================================
TAU_CONF = 0.0
TAU_EFF = 0.0
DELTA_TIE = 0.02
MAX_ITER = 10

TYPE_WEIGHT = {
    "causes": 1.0,
    "precedes": 0.7,
    "elaborates": 0.35,  # note: elaborates is flipped direction
}

SKIP_TYPES = {"contrasts_with", "conflicts_with"}  # do not enter graph
UNIFIED_PRED = "EPISODE_CAUSAL_LINK"


# ============================================================
# Utilities
# ============================================================
def _norm_type(x):
    return str(x or "").strip().lower()


def _safe_float(x, default=1.0):
    try:
        return float(x)
    except Exception:
        return float(default)


def _node_info(G, nid):
    d = dict(G.nodes[nid]) if nid in G.nodes else {}
    return {
        "id": nid,
        "name": d.get("name"),
        "description": d.get("description"),
        "properties": d.get("properties", {}),
    }


def _edge_info(G, u, v):
    d = dict(G[u][v]) if G.has_edge(u, v) else {}
    return {
        "from": u,
        "to": v,
        "predicate": d.get("predicate"),
        "relation_type": d.get("relation_type"),
        "confidence": d.get("confidence"),
        "type_weight": d.get("type_weight"),
        "effective_weight": d.get("effective_weight"),
        "description": d.get("description"),
        "properties": d.get("properties", {}),
        "evidence_pool": d.get("evidence_pool", []),
    }


def _parse_llm_json(s):
    if isinstance(s, dict):
        return s
    if not isinstance(s, str):
        return {}
    ss = s.strip()
    if not ss:
        return {}
    return json.loads(ss)


def _edge_cost_for_cycle_break(d):
    # cost smaller => remove earlier
    ew = _safe_float(d.get("effective_weight", 0.0), 0.0)
    return 1.0 - ew


# ============================================================
# 2) Normalize relations and build graph
# ============================================================
def build_causal_graph(V, R, *, tau_conf=TAU_CONF, tau_eff=TAU_EFF):
    """
    V: list of episode nodes dicts, must include id
    R: list of relations dicts:
       subject_id/subject, object_id/object, relation_type/predicate, confidence, description
    """

    G = nx.DiGraph()

    # nodes
    for e in (V or []):
        if not isinstance(e, dict):
            continue
        eid = e.get("id")
        if not eid:
            continue
        G.add_node(
            eid,
            name=e.get("name"),
            description=e.get("description"),
            properties=e.get("properties", {}),
        )

    # keep best edge per (u,v), merge the rest into evidence_pool
    best = {}  # (u,v) -> attr dict

    for r in (R or []):
        if not isinstance(r, dict):
            continue

        s = r.get("subject_id") or r.get("subject")
        o = r.get("object_id") or r.get("object")
        if not s or not o or s == o:
            continue
        if s not in G.nodes or o not in G.nodes:
            continue

        rel_type = _norm_type(r.get("relation_type") or r.get("predicate"))
        if not rel_type:
            continue
        if rel_type in SKIP_TYPES:
            continue
        if rel_type not in TYPE_WEIGHT:
            continue

        conf = _safe_float(r.get("confidence", 1.0), 1.0)
        if conf < float(tau_conf):
            continue

        # elaborates: flip direction A elaborates B => B -> A
        u, v = (o, s) if rel_type == "elaborates" else (s, o)

        type_w = float(TYPE_WEIGHT[rel_type])
        eff = conf * type_w
        if eff < float(tau_eff):
            continue

        attr = {
            "predicate": UNIFIED_PRED,
            "relation_type": rel_type,
            "confidence": conf,
            "type_weight": type_w,
            "effective_weight": eff,
            "description": r.get("description", ""),
            "properties": r.get("properties", {}) if isinstance(r.get("properties", {}), dict) else {},
            "evidence_pool": [],
        }

        key = (u, v)
        if key not in best:
            best[key] = attr
        else:
            if float(attr["effective_weight"]) > float(best[key]["effective_weight"]):
                best[key]["evidence_pool"].append(dict(best[key]))
                best[key] = attr
            else:
                best[key]["evidence_pool"].append(dict(attr))

    for (u, v), attr in best.items():
        if u in G.nodes and v in G.nodes and u != v:
            G.add_edge(u, v, **attr)

    return G


# ============================================================
# 4) Stage I: SCC break cycles (effective-weight baseline)
# ============================================================
def stage1_break_scc_cycles(
    G,
    *,
    delta_tie=DELTA_TIE,
    enable_llm=False,
    llm_choose_edge_fn=None,
    iter_id=1,
    log=None,
    show_progress=False,
):
    """
    Break cycles inside SCCs by removing lowest-cost edges, cost = 1 - effective_weight.

    LLM is optional and only used when:
      - enable_llm is True
      - llm_choose_edge_fn is provided
      - tie among best-cost edges within delta_tie

    llm_choose_edge_fn signature:
      (G, scc_nodes_list, tie_edges_list) -> (u,v) selected to remove
      where tie_edges_list = [(u,v,edge_attr,cost), ...]
    """
    if log is None:
        log = []

    removed_any = False

    def _scc_list():
        return [set(c) for c in nx.strongly_connected_components(G) if len(c) > 1]

    outer = range(10**9)  # loop until no SCC>1
    if show_progress:
        outer = tqdm(outer, desc=f"Stage1 SCC break iter={iter_id}", unit="round")

    for _ in outer:
        sccs = _scc_list()
        if not sccs:
            break

        for comp in sccs:
            # keep removing until this component becomes acyclic (it may split; outer loop will refresh)
            while True:
                S = G.subgraph(comp).copy()
                if nx.is_directed_acyclic_graph(S):
                    break

                edges = []
                for u, v, d in S.edges(data=True):
                    cost = float(_edge_cost_for_cycle_break(d))
                    edges.append((u, v, dict(d), cost))
                if not edges:
                    break

                edges.sort(key=lambda x: x[3])  # cost asc
                best_cost = edges[0][3]
                tie = [e for e in edges if abs(e[3] - best_cost) < float(delta_tie)]

                if enable_llm and llm_choose_edge_fn is not None and len(tie) > 1:
                    pick = llm_choose_edge_fn(G, list(comp), tie)
                    if pick is None:
                        u, v, d, cost = edges[0]
                    else:
                        u, v = pick
                        d = dict(G[u][v]) if G.has_edge(u, v) else {}
                        cost = float(_edge_cost_for_cycle_break(d))
                else:
                    u, v, d, cost = edges[0]

                if G.has_edge(u, v):
                    G.remove_edge(u, v)
                    removed_any = True
                    log.append(
                        {
                            "iter": iter_id,
                            "stage": "SCC_BREAK",
                            "action": "remove_edge",
                            "edge": {"from": u, "to": v},
                            "relation_type": d.get("relation_type"),
                            "effective_weight": d.get("effective_weight"),
                            "cost": cost,
                            "reason": "break_cycle_in_scc",
                        }
                    )

    if show_progress:
        outer.close()

    return removed_any, log


# ============================================================
# 5) Stage II: Triangle pruning (LLM-first)
# ============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from tqdm import tqdm


def stage2_triangle_prune_llm_first(
    G,
    *,
    narrative_manager,
    enable_llm=True,

    max_llm_calls=None,
    iter_id=1,
    log=None,
    show_progress=True,
):
    """
    LLM-first triangle pruning, with 64-thread concurrent LLM calls.

    For each triangle A->B, B->C, A->C:
      - If all three edges are causes: ALWAYS call LLM to decide remove A->C or keep.
      - Else strong-cause protection:
          If A->C is causes and at least one of A->B or B->C is NOT causes:
              auto keep A->C (no LLM).
      - Otherwise:
          call LLM to decide remove A->C or keep.

    Notes:
      - LLM calls are executed concurrently using ThreadPoolExecutor(max_workers=64).
      - max_llm_calls limits the number of triangles sent to LLM per call.
      - We still pass effective_weight etc. to LLM via relation_* payload; but decision is LLM-driven.
    """
    if log is None:
        log = []

    if not enable_llm or narrative_manager is None:
        raise ValueError("LLM-first Stage2 requires narrative_manager and enable_llm=True")

    stats = {
        "triangles_total": 0,
        "llm_candidates": 0,
        "llm_called": 0,
        "llm_remove": 0,
        "llm_keep": 0,
        "auto_keep_strong_cause": 0,
        "marked_remove": 0,
        "removed": 0,
    }

    def _norm_type(x):
        return str(x or "").strip().lower()

    def _node_info(nid):
        d = dict(G.nodes[nid]) if nid in G.nodes else {}
        return {
            "id": nid,
            "name": d.get("name"),
            "description": d.get("description"),
            "properties": d.get("properties", {}),
        }

    def _edge_info(u, v):
        d = dict(G[u][v]) if G.has_edge(u, v) else {}
        return {
            "from": u,
            "to": v,
            "predicate": d.get("predicate"),
            "relation_type": d.get("relation_type"),
            "confidence": d.get("confidence"),
            "type_weight": d.get("type_weight"),
            "effective_weight": d.get("effective_weight"),
            "description": d.get("description"),
            "properties": d.get("properties", {}),
            "evidence_pool": d.get("evidence_pool", []),
        }

    def _rtype(u, v):
        return _norm_type(G[u][v].get("relation_type"))

    # 1) Enumerate triangles, collect LLM jobs (A,B,C) where needed
    llm_jobs = []  # list of (a,b,c,t_ab,t_bc,t_ac)
    nodes = list(G.nodes)

    it = nodes
    if show_progress:
        it = tqdm(nodes, desc=f"Stage2 collect triangles iter={iter_id}", unit="node")

    for a in it:
        for b in list(G.successors(a)):
            for c in list(G.successors(b)):
                if c == a or c == b:
                    continue
                if not G.has_edge(a, c):
                    continue

                stats["triangles_total"] += 1

                t_ab = _rtype(a, b)
                t_bc = _rtype(b, c)
                t_ac = _rtype(a, c)
                all_causes = (t_ab == "causes" and t_bc == "causes" and t_ac == "causes")

                # strong cause protection: keep A->C without LLM
                if (t_ac == "causes") and (not all_causes) and (t_ab != "causes" or t_bc != "causes"):
                    stats["auto_keep_strong_cause"] += 1
                    log.append(
                        {
                            "iter": iter_id,
                            "stage": "TRIANGLE",
                            "action": "auto_keep",
                            "rule": "STRONG_CAUSE_PROTECT",
                            "edge": {"from": a, "to": c},
                            "types": {"ab": t_ab, "bc": t_bc, "ac": t_ac},
                            "reason": "A->C is causes but chain has non-causes edge(s)",
                        }
                    )
                    continue

                llm_jobs.append((a, b, c, t_ab, t_bc, t_ac))

        if show_progress:
            it.set_postfix(
                tri=stats["triangles_total"],
                llm_cand=len(llm_jobs),
                auto_keep=stats["auto_keep_strong_cause"],
            )

    if show_progress:
        it.close()

    # Apply max_llm_calls cap
    stats["llm_candidates"] = len(llm_jobs)
    if max_llm_calls is not None:
        llm_jobs = llm_jobs[: int(max_llm_calls)]

    if not llm_jobs:
        return stats, log

    # 2) Run LLM calls concurrently (64 threads)
    to_remove = set()

    def _call_one(job):
        a, b, c, t_ab, t_bc, t_ac = job
        out = narrative_manager.prune_causal_edge(
            entity_a=_node_info(a),
            entity_b=_node_info(b),
            entity_c=_node_info(c),
            relation_ab=_edge_info(a, b),
            relation_bc=_edge_info(b, c),
            relation_ac=_edge_info(a, c),

        )
        parsed = json.loads(out) if isinstance(out, str) else (out or {})
        remove_edge = bool(parsed.get("remove_edge", False))
        reason = parsed.get("reason", "")
        return (a, b, c, t_ab, t_bc, t_ac, remove_edge, reason)

    futures = []
    with ThreadPoolExecutor(max_workers=64) as ex:
        for job in llm_jobs:
            futures.append(ex.submit(_call_one, job))

        it2 = as_completed(futures)
        if show_progress:
            it2 = tqdm(it2, total=len(futures), desc=f"Stage2 LLM prune iter={iter_id}", unit="tri")

        for fut in it2:
            try:
                a, b, c, t_ab, t_bc, t_ac, remove_edge, reason = fut.result()
            except Exception as e:
                # log and skip
                log.append(
                    {
                        "iter": iter_id,
                        "stage": "TRIANGLE",
                        "action": "llm_error",
                        "error": str(e),
                    }
                )
                continue

            stats["llm_called"] += 1

            if remove_edge:
                stats["llm_remove"] += 1
                stats["marked_remove"] += 1
                to_remove.add((a, c))
                log.append(
                    {
                        "iter": iter_id,
                        "stage": "TRIANGLE",
                        "action": "llm_mark_remove",
                        "edge": {"from": a, "to": c},
                        "types": {"ab": t_ab, "bc": t_bc, "ac": t_ac},
                        "reason": reason,
                    }
                )
            else:
                stats["llm_keep"] += 1
                log.append(
                    {
                        "iter": iter_id,
                        "stage": "TRIANGLE",
                        "action": "llm_keep",
                        "edge": {"from": a, "to": c},
                        "types": {"ab": t_ab, "bc": t_bc, "ac": t_ac},
                        "reason": reason,
                    }
                )

        if show_progress:
            it2.close()

    # 3) Apply removals
    for u, v in to_remove:
        if G.has_edge(u, v):
            G.remove_edge(u, v)
            stats["removed"] += 1
            log.append(
                {
                    "iter": iter_id,
                    "stage": "TRIANGLE",
                    "action": "remove_edge",
                    "edge": {"from": u, "to": v},
                    "reason": "triangle_prune_llm_first",
                }
            )

    return stats, log


# ============================================================
# 6) SABER main loop (Stage II first, then Stage I)
# ============================================================
def run_saber_llm_first(V, R, *, narrative_manager, max_iter=MAX_ITER, max_llm_calls_per_iter=200, show_progress=True):
    """
    SABER (LLM-first triangles):
      For each iter:
        Stage II: triangle prune (LLM-first)
        Stage I: SCC break cycles (effective-weight, no LLM by default)
        Converge when:
          - no SCC > 1
          - and Stage II removed 0
          - and Stage I removed 0
    """
    G = build_causal_graph(V, R, tau_conf=TAU_CONF, tau_eff=TAU_EFF)
    log = []

    for it in range(1, int(max_iter) + 1):
        stats_tri, log = stage2_triangle_prune_llm_first(
            G,
            narrative_manager=narrative_manager,
            enable_llm=True,

            max_llm_calls=max_llm_calls_per_iter,
            iter_id=it,
            log=log,
            show_progress=show_progress,
        )

        removed_scc, log = stage1_break_scc_cycles(
            G,
            delta_tie=DELTA_TIE,
            enable_llm=False,
            llm_choose_edge_fn=None,
            iter_id=it,
            log=log,
            show_progress=False,
        )

        has_scc = any(len(c) > 1 for c in nx.strongly_connected_components(G))
        removed_tri = bool(stats_tri.get("removed", 0) > 0)

        if (not has_scc) and (not removed_scc) and (not removed_tri):
            log.append({"iter": it, "stage": "CONVERGE", "action": "break", "reason": "no_scc_and_no_removals"})
            break

    return G, log


# ============================================================
# Baseline: effective-weight only (Stage I only)
# ============================================================
def run_effective_weight_baseline(V, R, *, max_iter=MAX_ITER):
    G = build_causal_graph(V, R, tau_conf=TAU_CONF, tau_eff=TAU_EFF)
    log = []
    for it in range(1, int(max_iter) + 1):
        removed, log = stage1_break_scc_cycles(
            G,
            delta_tie=DELTA_TIE,
            enable_llm=False,
            llm_choose_edge_fn=None,
            iter_id=it,
            log=log,
            show_progress=False,
        )
        has_scc = any(len(c) > 1 for c in nx.strongly_connected_components(G))
        if (not has_scc) or (not removed):
            log.append({"iter": it, "stage": "CONVERGE", "action": "break", "reason": "no_scc_or_no_removal"})
            break
    return G, log


# ============================================================
# Quick diagnostics helpers
# ============================================================
def count_sccs(G):
    sccs = [c for c in nx.strongly_connected_components(G) if len(c) > 1]
    return len(sccs), sorted([len(c) for c in sccs], reverse=True)


def graph_summary(G, name="G"):
    n_scc, sizes = count_sccs(G)
    print(f"{name}: nodes={G.number_of_nodes()} edges={G.number_of_edges()} dag={nx.is_directed_acyclic_graph(G)} sccs={n_scc} top_sizes={sizes[:10]}")



In [2]:
import numpy as np

def build_embedding_map_from_episodes(
    episodes,
    *,
    id_key="id",
    emb_key="embedding",
    as_numpy=True,
    dtype=np.float32,
    min_dim=8,
):
    """
    episodes: list[dict] loaded from episodes.json, each has {id, embedding}
    returns: dict[id] -> vector (np.ndarray or list)
    """
    emb = {}
    missing = 0
    bad = 0
    for e in (episodes or []):
        if not isinstance(e, dict):
            continue
        eid = e.get(id_key)
        vec = e.get(emb_key)
        if not eid:
            continue
        if vec is None:
            missing += 1
            continue
        if not isinstance(vec, (list, tuple)) or len(vec) < int(min_dim):
            bad += 1
            continue
        if as_numpy:
            try:
                emb[eid] = np.asarray(vec, dtype=dtype)
            except Exception:
                bad += 1
                continue
        else:
            emb[eid] = list(vec)
    return emb, {"missing": missing, "bad": bad, "kept": len(emb)}


In [3]:
episodes = json.load(open("data/narrative_graph/global/episodes.json", "r", encoding="utf-8"))
relations = json.load(open("data/narrative_graph/global/episode_relations.json", "r", encoding="utf-8"))

from core.model_providers.openai_llm import OpenAILLM
from core import KAGConfig
from core.builder.manager.narrative_analysis_manager import NarrativeManager

config = KAGConfig.from_yaml("configs/config_openai.yaml")
llm = OpenAILLM(config)
narrative_manager = NarrativeManager(config, llm)

# SABER (LLM-first triangles)
G_saber, saber_log = run_saber_llm_first(
    episodes,
    relations,
    narrative_manager=narrative_manager,
    max_iter=10,
    max_llm_calls_per_iter=200,
    show_progress=True
)
graph_summary(G_saber, "SABER")
print("SABER log size:", len(saber_log))

# Baseline (effective-weight only)
G_base, base_log = run_effective_weight_baseline(episodes, relations, max_iter=10)
graph_summary(G_base, "BASE")
print("BASE log size:", len(base_log))

/vepfs-mlp2/c20250513/241404044/users/roytian/anaconda3/envs/screenplay/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Stage2 LLM prune iter=3: 100%|██████████| 56/56 [00:06<00:00,  8.75tri/s]


SABER: nodes=151 edges=193 dag=True sccs=0 top_sizes=[]
SABER log size: 508
BASE: nodes=151 edges=198 dag=True sccs=0 top_sizes=[]
BASE log size: 171


In [4]:
import json
import random
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import networkx as nx


# ============================================================
# Helpers
# ============================================================
def _safe_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return float(default)


def _cosine(a: np.ndarray, b: np.ndarray, eps: float = 1e-9) -> float:
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    num = float(np.dot(a, b))
    den = float(np.linalg.norm(a) * np.linalg.norm(b) + eps)
    return num / den


def _node_info(G: nx.DiGraph, nid: str) -> Dict[str, Any]:
    d = dict(G.nodes[nid]) if nid in G.nodes else {}
    return {
        "id": nid,
        "name": d.get("name"),
        "description": d.get("description"),
        "properties": d.get("properties", {}),
    }


def _edge_info(G: nx.DiGraph, u: str, v: str) -> Dict[str, Any]:
    d = dict(G[u][v]) if G.has_edge(u, v) else {}
    return {
        "from": u,
        "to": v,
        "predicate": d.get("predicate"),
        "relation_type": d.get("relation_type"),
        "confidence": d.get("confidence"),
        "type_weight": d.get("type_weight"),
        "effective_weight": d.get("effective_weight"),
        "description": d.get("description"),
        "properties": d.get("properties", {}),
        "evidence_pool": d.get("evidence_pool", []),
    }


def _try_parse_json(text: Any) -> Dict[str, Any]:
    if isinstance(text, dict):
        return text
    if not isinstance(text, str):
        return {}
    s = text.strip()
    if not s:
        return {}
    # Try raw json
    try:
        return json.loads(s)
    except Exception:
        pass
    # Try to extract first {...}
    l = s.find("{")
    r = s.rfind("}")
    if l >= 0 and r > l:
        try:
            return json.loads(s[l : r + 1])
        except Exception:
            return {}
    return {}


# ============================================================
# Metric 1 + 2: Shortcut ratio + Transitive reduction stats
#   If DAG: shortcut_edges = E - E_TR (exact, cheap, reliable)
# ============================================================
def transitive_reduction_metrics(G: nx.DiGraph) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "is_dag": nx.is_directed_acyclic_graph(G),
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
    }
    if not out["is_dag"]:
        out["error"] = "Graph is not a DAG. Transitive reduction metrics require DAG."
        return out

    TR = nx.algorithms.dag.transitive_reduction(G)
    E = set(G.edges())
    Etr = set(TR.edges())
    shortcut = list(E - Etr)

    out.update(
        {
            "edges_tr": len(Etr),
            "reduction_rate": 1.0 - (len(Etr) / max(1, len(E))),
            "shortcut_edges": len(shortcut),
            "shortcut_ratio": len(shortcut) / max(1, len(E)),
        }
    )
    return out


# ============================================================
# Metric 4: Path smoothness using embeddings
#   Sample paths of fixed length k (edges), compute mean and var
# ============================================================
def path_smoothness(
    G: nx.DiGraph,
    emb: Dict[str, Any],
    *,
    path_len_edges: int = 3,  # number of edges, nodes = edges+1
    num_paths: int = 2000,
    seed: int = 0,
) -> Dict[str, Any]:
    rng = random.Random(seed)
    if path_len_edges < 2:
        raise ValueError("path_len_edges must be >= 2")

    # Only keep nodes that have embedding
    nodes = [n for n in G.nodes() if n in emb]
    if not nodes:
        return {"error": "No nodes with embeddings found."}

    def _one_path() -> Optional[List[str]]:
        # pick a start that can walk enough steps
        for _ in range(30):
            cur = rng.choice(nodes)
            path = [cur]
            ok = True
            for _step in range(path_len_edges):
                succ = [x for x in G.successors(cur) if x in emb]
                if not succ:
                    ok = False
                    break
                cur = rng.choice(succ)
                path.append(cur)
            if ok:
                return path
        return None

    path_stats = []
    for _ in range(int(num_paths)):
        p = _one_path()
        if not p:
            continue
        sims = []
        for i in range(len(p) - 1):
            sims.append(_cosine(emb[p[i]], emb[p[i + 1]]))
        path_stats.append((float(np.mean(sims)), float(np.var(sims))))

    if not path_stats:
        return {"error": "Failed to sample any valid paths with embeddings."}

    means = [x[0] for x in path_stats]
    vars_ = [x[1] for x in path_stats]
    return {
        "path_len_edges": path_len_edges,
        "num_paths_requested": num_paths,
        "num_paths_used": len(path_stats),
        "smooth_mean_mean": float(np.mean(means)),
        "smooth_mean_median": float(np.median(means)),
        "smooth_var_mean": float(np.mean(vars_)),
        "smooth_var_median": float(np.median(vars_)),
    }


# ============================================================
# Metric 5: Edge semantic coherence vs non-edge
# ============================================================
def edge_coherence_vs_nonedge(
    G: nx.DiGraph,
    emb: Dict[str, Any],
    *,
    neg_multiplier: float = 1.0,
    max_pos: Optional[int] = None,
    seed: int = 0,
) -> Dict[str, Any]:
    rng = random.Random(seed)

    pos_edges = [(u, v) for (u, v) in G.edges() if u in emb and v in emb]
    if not pos_edges:
        return {"error": "No edges with both endpoints having embeddings."}

    if max_pos is not None and max_pos < len(pos_edges):
        pos_edges = rng.sample(pos_edges, int(max_pos))

    pos = [_cosine(emb[u], emb[v]) for (u, v) in pos_edges]

    nodes = [n for n in G.nodes() if n in emb]
    target_neg = int(len(pos) * float(neg_multiplier))
    neg = []
    tried = 0
    # bounded trials to avoid infinite loops on dense graphs
    max_trials = max(10000, target_neg * 50)

    while len(neg) < target_neg and tried < max_trials:
        tried += 1
        u = rng.choice(nodes)
        v = rng.choice(nodes)
        if u == v:
            continue
        if G.has_edge(u, v):
            continue
        neg.append(_cosine(emb[u], emb[v]))

    out = {
        "pos_n": len(pos),
        "neg_n": len(neg),
        "pos_mean": float(np.mean(pos)),
        "pos_median": float(np.median(pos)),
        "neg_mean": float(np.mean(neg)) if neg else None,
        "neg_median": float(np.median(neg)) if neg else None,
        "gap_mean": (float(np.mean(pos)) - float(np.mean(neg))) if neg else None,
    }
    return out


# ============================================================
# Metric 3: LLM preference on edge diffs
#   Uses llm.run(messages) where messages=[{"role":"user","content":prompt}]
#   Judges whether an edge should exist as a direct causal/ordering link.
# ============================================================
def _edge_judge_prompt(
    *,
    entity_u: Dict[str, Any],
    entity_v: Dict[str, Any],
    edge_uv: Dict[str, Any],
) -> str:
    # Keep prompt deterministic and JSON-only output
    return (
        "You are judging whether a directed edge between two Episodes should exist as a direct link "
        "in a causal or temporal narrative graph.\n\n"
        "Return JSON only.\n\n"
        "Decision rule:\n"
        "- keep: the edge is a necessary direct relation (cannot be easily replaced by a more specific intermediate episode) and is plausible.\n"
        "- remove: the edge is likely redundant, a shortcut, too vague, or better expressed via intermediate steps.\n\n"
        "Output JSON schema:\n"
        "{{\n"
        '  "decision": "keep" | "remove",\n'
        '  "confidence": number,  \n'
        '  "reason": string\n'
        "}}\n\n"
        "Episode U:\n"
        f"{json.dumps(entity_u, ensure_ascii=False, indent=2)}\n\n"
        "Episode V:\n"
        f"{json.dumps(entity_v, ensure_ascii=False, indent=2)}\n\n"
        "Proposed edge U -> V:\n"
        f"{json.dumps(edge_uv, ensure_ascii=False, indent=2)}\n"
    )


def llm_preference_on_edge_diffs(
    llm,
    G_saber: nx.DiGraph,
    G_base: nx.DiGraph,
    *,
    sample_each: int = 200,
    seed: int = 0,
) -> Dict[str, Any]:
    rng = random.Random(seed)

    E_s = set(G_saber.edges())
    E_b = set(G_base.edges())
    saber_only = list(E_s - E_b)
    base_only = list(E_b - E_s)

    if sample_each is not None:
        if len(saber_only) > sample_each:
            saber_only = rng.sample(saber_only, sample_each)
        if len(base_only) > sample_each:
            base_only = rng.sample(base_only, sample_each)

    def _judge_one(G: nx.DiGraph, u: str, v: str) -> Dict[str, Any]:
        prompt = _edge_judge_prompt(
            entity_u=_node_info(G, u),
            entity_v=_node_info(G, v),
            edge_uv=_edge_info(G, u, v),
        )
        raw = llm.run(messages=[{"role": "user", "content": prompt}])
        parsed = _try_parse_json(raw)

        decision = str(parsed.get("decision", "")).strip().lower()
        if decision not in ("keep", "remove"):
            # fallback: map boolean-like fields if present
            if bool(parsed.get("keep_edge", False)) is True:
                decision = "keep"
            elif bool(parsed.get("remove_edge", False)) is True:
                decision = "remove"
            else:
                decision = "remove"

        conf = _safe_float(parsed.get("confidence", 0.5), 0.5)
        reason = parsed.get("reason", "")

        return {
            "edge": {"from": u, "to": v},
            "decision": decision,
            "confidence": conf,
            "reason": reason,
        }

    def _run_group(name: str, edges: List[Tuple[str, str]], G: nx.DiGraph) -> Dict[str, Any]:
        rows = []
        keep = 0
        remove = 0
        confs = []

        for (u, v) in edges:
            if not G.has_edge(u, v):
                continue
            r = _judge_one(G, u, v)
            rows.append(r)
            if r["decision"] == "keep":
                keep += 1
            else:
                remove += 1
            confs.append(float(r["confidence"]))

        return {
            "name": name,
            "n": len(rows),
            "keep_rate": (keep / max(1, len(rows))),
            "remove_rate": (remove / max(1, len(rows))),
            "avg_confidence": float(np.mean(confs)) if confs else None,
            "rows": rows,  # you can drop this if too large
        }

    return {
        "diff_sizes": {
            "saber_only_total": len(set(E_s - E_b)),
            "base_only_total": len(set(E_b - E_s)),
            "intersection": len(E_s & E_b),
        },
        "samples": {"sample_each": sample_each, "seed": seed},
        "saber_only_eval": _run_group("saber_only", saber_only, G_saber),
        "base_only_eval": _run_group("base_only", base_only, G_base),
    }


# ============================================================
# One-shot runner: compute the selected metrics
# ============================================================
def evaluate_causal_graphs(
    G_saber: nx.DiGraph,
    G_base: nx.DiGraph,
    *,
    emb: Optional[Dict[str, Any]] = None,
    llm: Optional[Any] = None,
    llm_sample_each: int = 200,
    seed: int = 0,
) -> Dict[str, Any]:
    out: Dict[str, Any] = {"seed": seed}

    # Structural: transitive reduction, shortcut ratio
    out["saber_tr"] = transitive_reduction_metrics(G_saber)
    out["base_tr"] = transitive_reduction_metrics(G_base)

    # Embedding metrics
    if emb is not None:
        out["saber_path_smoothness"] = path_smoothness(G_saber, emb, path_len_edges=3, num_paths=2000, seed=seed)
        out["base_path_smoothness"] = path_smoothness(G_base, emb, path_len_edges=3, num_paths=2000, seed=seed + 1)
        out["saber_edge_coherence"] = edge_coherence_vs_nonedge(G_saber, emb, neg_multiplier=1.0, max_pos=5000, seed=seed)
        out["base_edge_coherence"] = edge_coherence_vs_nonedge(G_base, emb, neg_multiplier=1.0, max_pos=5000, seed=seed + 1)

    # LLM preference on edge diffs
    if llm is not None:
        out["llm_preference"] = llm_preference_on_edge_diffs(
            llm,
            G_saber,
            G_base,
            sample_each=llm_sample_each,
            seed=seed,
        )

    return out



In [5]:
G_base.nodes["ep_8b028a56b63b4a9e"]

{'name': 'Approach the harbor',
 'description': "Berlin's Mercedes descends the coast road toward the harbor, marking the arrival at a coastal setting featuring sand dunes, a beach, and a lagoon behind a sea-break. This episode captures the initial movement into the scene, establishing the location and direction of travel as a goal-driven action of reaching the harbor area.",
 'properties': {'doc_id': 'doc_37',
  'related_events': ['Mercedes descends coast road toward harbor (scene_38)']}}

In [6]:
emb, emb_stats = build_embedding_map_from_episodes(episodes)
print("embedding map stats:", emb_stats)

metrics = evaluate_causal_graphs(
    G_saber,
    G_base,
    emb=emb,     # 直接用 episodes.json 的 embedding
    llm=llm,     # 你要跑 LLM diff 评估就传，不跑就设 None
    llm_sample_each=200,
    seed=0,
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))


embedding map stats: {'missing': 0, 'bad': 0, 'kept': 151}
{
  "seed": 0,
  "saber_tr": {
    "is_dag": true,
    "nodes": 151,
    "edges": 193,
    "edges_tr": 150,
    "reduction_rate": 0.2227979274611399,
    "shortcut_edges": 43,
    "shortcut_ratio": 0.22279792746113988
  },
  "base_tr": {
    "is_dag": true,
    "nodes": 151,
    "edges": 198,
    "edges_tr": 151,
    "reduction_rate": 0.23737373737373735,
    "shortcut_edges": 47,
    "shortcut_ratio": 0.23737373737373738
  },
  "saber_path_smoothness": {
    "path_len_edges": 3,
    "num_paths_requested": 2000,
    "num_paths_used": 1919,
    "smooth_mean_mean": 0.6566712735199938,
    "smooth_mean_median": 0.6544725100199381,
    "smooth_var_mean": 0.001159798874417193,
    "smooth_var_median": 0.00043525101130953095
  },
  "base_path_smoothness": {
    "path_len_edges": 3,
    "num_paths_requested": 2000,
    "num_paths_used": 1919,
    "smooth_mean_mean": 0.6540226164077289,
    "smooth_mean_median": 0.6526150626095067,
   

In [7]:
import json
import random
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import networkx as nx


# ============================================================
# Core math + utils
# ============================================================
def _safe_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return float(default)


def _cosine(a: Any, b: Any, eps: float = 1e-9) -> float:
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    num = float(np.dot(a, b))
    den = float(np.linalg.norm(a) * np.linalg.norm(b) + eps)
    return num / den


def _node_info(G: nx.DiGraph, nid: str) -> Dict[str, Any]:
    d = dict(G.nodes[nid]) if nid in G.nodes else {}
    return {
        "id": nid,
        "name": d.get("name"),
        "description": d.get("description"),
        "properties": d.get("properties", {}),
    }


def _edge_info(G: nx.DiGraph, u: str, v: str) -> Dict[str, Any]:
    d = dict(G[u][v]) if G.has_edge(u, v) else {}
    return {
        "from": u,
        "to": v,
        "predicate": d.get("predicate"),
        "relation_type": d.get("relation_type"),
        "confidence": d.get("confidence"),
        "type_weight": d.get("type_weight"),
        "effective_weight": d.get("effective_weight"),
        "description": d.get("description"),
        "properties": d.get("properties", {}),
        "evidence_pool": d.get("evidence_pool", []),
    }


def _extract_text_from_llm_run_output(raw: Any) -> str:
    """
    OpenAILLM.llm.run(messages) may return:
      - str
      - dict-like with 'content'
      - Message object with .content
      - list/tuple of Message/dict/str (often [Message(...)] or [Message(...), ...])

    This normalizes to a single string for JSON parsing.
    """
    if raw is None:
        return ""

    # Plain string
    if isinstance(raw, str):
        return raw

    # List/tuple: take the last assistant-like content if possible, else join
    if isinstance(raw, (list, tuple)):
        texts = []
        for item in raw:
            texts.append(_extract_text_from_llm_run_output(item))
        # Heuristic: return last non-empty
        for t in reversed(texts):
            if isinstance(t, str) and t.strip():
                return t
        return "\n".join([t for t in texts if isinstance(t, str)])

    # Dict-like
    if isinstance(raw, dict):
        c = raw.get("content")
        if isinstance(c, str):
            return c
        # Sometimes nested
        if c is not None:
            return _extract_text_from_llm_run_output(c)
        return json.dumps(raw, ensure_ascii=False)

    # Object with content attribute
    if hasattr(raw, "content"):
        try:
            c = getattr(raw, "content")
            if isinstance(c, str):
                return c
            return _extract_text_from_llm_run_output(c)
        except Exception:
            pass

    # Fallback
    return str(raw)


def _try_parse_json(text: Any) -> Dict[str, Any]:
    """
    Accepts:
      - dict
      - str containing JSON
      - other LLM outputs => will attempt to parse JSON substring
    """
    if isinstance(text, dict):
        return text

    s = text if isinstance(text, str) else _extract_text_from_llm_run_output(text)
    s = (s or "").strip()
    if not s:
        return {}

    # Try raw JSON first
    try:
        return json.loads(s)
    except Exception:
        pass

    # Try extracting first {...} block
    l = s.find("{")
    r = s.rfind("}")
    if l >= 0 and r > l:
        try:
            return json.loads(s[l : r + 1])
        except Exception:
            return {}

    return {}


def _rtype(G: nx.DiGraph, u: str, v: str) -> str:
    if not G.has_edge(u, v):
        return ""
    return str(G[u][v].get("relation_type") or "").strip().lower()


# ============================================================
# Metric 1 + 2: Transitive reduction (exact shortcut ratio on DAG)
# ============================================================
def transitive_reduction_metrics(G: nx.DiGraph) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "is_dag": nx.is_directed_acyclic_graph(G),
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
    }
    if not out["is_dag"]:
        out["error"] = "Graph is not a DAG. Transitive reduction requires DAG."
        return out

    TR = nx.algorithms.dag.transitive_reduction(G)
    E = set(G.edges())
    Etr = set(TR.edges())
    shortcut_edges = list(E - Etr)

    out.update(
        {
            "edges_tr": len(Etr),
            "reduction_rate": 1.0 - (len(Etr) / max(1, len(E))),
            "shortcut_edges": len(shortcut_edges),
            "shortcut_ratio": len(shortcut_edges) / max(1, len(E)),
        }
    )
    return out


# ============================================================
# Improved Metric A: Matched negative sampling edge coherence
# ============================================================
def edge_coherence_matched_negatives(
    G: nx.DiGraph,
    emb: Dict[str, Any],
    *,
    max_pos: Optional[int] = None,
    seed: int = 0,
    per_type: bool = True,
) -> Dict[str, Any]:
    rng = random.Random(seed)

    pos_edges_all = [(u, v) for (u, v) in G.edges() if u in emb and v in emb]
    if not pos_edges_all:
        return {"error": "No edges with both endpoints having embeddings."}

    pos_edges = pos_edges_all
    if max_pos is not None and max_pos < len(pos_edges_all):
        pos_edges = rng.sample(pos_edges_all, int(max_pos))

    nodes_with_emb = [n for n in G.nodes() if n in emb]
    nodes_with_emb_set = set(nodes_with_emb)

    def _sample_neg_for_u(u: str, tries: int = 80) -> Optional[Tuple[str, str]]:
        for _ in range(tries):
            v2 = rng.choice(nodes_with_emb)
            if v2 == u:
                continue
            if v2 not in nodes_with_emb_set:
                continue
            if G.has_edge(u, v2):
                continue
            return (u, v2)
        return None

    pos_sims: List[float] = []
    neg_sims: List[float] = []
    bucket: Dict[str, Dict[str, List[float]]] = {}

    for (u, v) in pos_edges:
        neg = _sample_neg_for_u(u)
        if neg is None:
            continue
        u2, v2 = neg

        sp = _cosine(emb[u], emb[v])
        sn = _cosine(emb[u2], emb[v2])

        pos_sims.append(sp)
        neg_sims.append(sn)

        if per_type:
            t = _rtype(G, u, v) or "unknown"
            if t not in bucket:
                bucket[t] = {"pos": [], "neg": []}
            bucket[t]["pos"].append(sp)
            bucket[t]["neg"].append(sn)

    if not pos_sims:
        return {"error": "Failed to sample matched negatives for any edge."}

    out = {
        "pos_n": len(pos_sims),
        "neg_n": len(neg_sims),
        "pairs_used": len(pos_sims),
        "pos_mean": float(np.mean(pos_sims)),
        "pos_median": float(np.median(pos_sims)),
        "neg_mean": float(np.mean(neg_sims)),
        "neg_median": float(np.median(neg_sims)),
        "gap_mean": float(np.mean(pos_sims) - np.mean(neg_sims)),
        "gap_median": float(np.median(pos_sims) - np.median(neg_sims)),
        "sampling": {"strategy": "matched_by_source_u", "max_pos": max_pos, "seed": seed},
    }

    if per_type:
        out["per_type"] = {
            t: {
                "n": len(d["pos"]),
                "pos_mean": float(np.mean(d["pos"])) if d["pos"] else None,
                "neg_mean": float(np.mean(d["neg"])) if d["neg"] else None,
                "gap_mean": float(np.mean(d["pos"]) - np.mean(d["neg"])) if d["pos"] and d["neg"] else None,
            }
            for t, d in bucket.items()
        }

    return out


# ============================================================
# Shared-path smoothness (note: same node sequence => same cosine values)
# ============================================================
def _sample_paths_from_graph(
    G: nx.DiGraph,
    nodes_with_emb: List[str],
    emb: Dict[str, Any],
    *,
    path_len_edges: int,
    num_paths: int,
    seed: int,
) -> List[List[str]]:
    rng = random.Random(seed)

    def _walk(start: str) -> Optional[List[str]]:
        cur = start
        path = [cur]
        for _ in range(path_len_edges):
            succ = [x for x in G.successors(cur) if x in emb]
            if not succ:
                return None
            cur = rng.choice(succ)
            path.append(cur)
        return path

    paths = []
    tries = 0
    max_tries = max(2000, num_paths * 30)

    while len(paths) < num_paths and tries < max_tries:
        tries += 1
        s = rng.choice(nodes_with_emb)
        p = _walk(s)
        if p is None:
            continue
        paths.append(p)

    return paths


def path_smoothness_shared_paths(
    G_a: nx.DiGraph,
    G_b: nx.DiGraph,
    emb: Dict[str, Any],
    *,
    path_len_edges: int = 3,
    num_paths: int = 2000,
    seed: int = 0,
) -> Dict[str, Any]:
    nodes_with_emb = [n for n in G_a.nodes() if n in emb and n in G_b.nodes()]
    if not nodes_with_emb:
        return {"error": "No shared nodes with embeddings across both graphs."}

    cand = _sample_paths_from_graph(
        G_a,
        nodes_with_emb,
        emb,
        path_len_edges=path_len_edges,
        num_paths=num_paths * 2,
        seed=seed,
    )

    shared = []
    for p in cand:
        ok = True
        for i in range(len(p) - 1):
            if not G_b.has_edge(p[i], p[i + 1]):
                ok = False
                break
        if ok:
            shared.append(p)
        if len(shared) >= num_paths:
            break

    if not shared:
        return {"error": "No shared paths found of the requested length."}

    def _stats(paths: List[List[str]]) -> Dict[str, Any]:
        means = []
        vars_ = []
        for p in paths:
            sims = [_cosine(emb[p[i]], emb[p[i + 1]]) for i in range(len(p) - 1)]
            means.append(float(np.mean(sims)))
            vars_.append(float(np.var(sims)))
        return {
            "num_paths_used": len(paths),
            "smooth_mean_mean": float(np.mean(means)),
            "smooth_mean_median": float(np.median(means)),
            "smooth_var_mean": float(np.mean(vars_)),
            "smooth_var_median": float(np.median(vars_)),
        }

    return {
        "path_len_edges": path_len_edges,
        "num_paths_requested": num_paths,
        "num_paths_shared": len(shared),
        "sampling": {"strategy": "sample_from_G_a_then_filter_by_G_b", "seed": seed},
        "graph_a": _stats(shared),
        "graph_b": _stats(shared),
    }


# ============================================================
# LLM metric: preference on edge diffs (BUGFIXED)
# ============================================================
def _edge_judge_prompt(
    *,
    entity_u: Dict[str, Any],
    entity_v: Dict[str, Any],
    edge_uv: Dict[str, Any],
) -> str:
    return (
        "You are judging whether a directed edge between two Episodes should exist as a direct link "
        "in a causal or temporal narrative graph.\n\n"
        "Return JSON only. Do not output any other text.\n\n"
        "Decision rubric:\n"
        "- keep: the edge expresses a direct, necessary relation.\n"
        "- remove: the edge is likely redundant (a shortcut), too vague, or better expressed via intermediate episodes.\n\n"
        "Output JSON schema:\n"
        "{\n"
        '  "decision": "keep" | "remove",\n'
        '  "confidence": number,\n'
        '  "reason": string\n'
        "}\n\n"
        "Episode U:\n"
        f"{json.dumps(entity_u, ensure_ascii=False, indent=2)}\n\n"
        "Episode V:\n"
        f"{json.dumps(entity_v, ensure_ascii=False, indent=2)}\n\n"
        "Proposed edge U -> V:\n"
        f"{json.dumps(edge_uv, ensure_ascii=False, indent=2)}\n"
    )


def llm_preference_on_edge_diffs(
    llm,
    G_saber: nx.DiGraph,
    G_base: nx.DiGraph,
    *,
    sample_each: int = 200,
    seed: int = 0,
    keep_rows: bool = True,
) -> Dict[str, Any]:
    rng = random.Random(seed)
    E_s = set(G_saber.edges())
    E_b = set(G_base.edges())
    saber_only_all = list(E_s - E_b)
    base_only_all = list(E_b - E_s)

    saber_only = saber_only_all[:]
    base_only = base_only_all[:]
    if sample_each is not None:
        if len(saber_only) > sample_each:
            saber_only = rng.sample(saber_only, sample_each)
        if len(base_only) > sample_each:
            base_only = rng.sample(base_only, sample_each)

    def _judge_one(G: nx.DiGraph, u: str, v: str) -> Dict[str, Any]:
        prompt = _edge_judge_prompt(
            entity_u=_node_info(G, u),
            entity_v=_node_info(G, v),
            edge_uv=_edge_info(G, u, v),
        )

        raw = llm.run(messages=[{"role": "user", "content": prompt}])

        # BUGFIX: normalize to assistant content string before parsing
        raw_text = _extract_text_from_llm_run_output(raw)
        parsed = _try_parse_json(raw_text)

        decision = str(parsed.get("decision", "")).strip().lower()
        if decision not in ("keep", "remove"):
            if bool(parsed.get("keep_edge", False)):
                decision = "keep"
            elif bool(parsed.get("remove_edge", False)):
                decision = "remove"
            else:
                decision = "remove"

        conf = _safe_float(parsed.get("confidence", 0.5), 0.5)
        reason = str(parsed.get("reason", "") or "")

        return {
            "edge": {"from": u, "to": v},
            "decision": decision,
            "confidence": conf,
            "reason": reason,
            # Keep raw only if parsing looks suspicious
            "raw": raw_text if (reason.strip() == "" and abs(conf - 0.5) < 1e-9) else None,
        }

    def _run_group(name: str, edges: List[Tuple[str, str]], G: nx.DiGraph) -> Dict[str, Any]:
        rows = []
        keep = 0
        remove = 0
        confs = []

        for (u, v) in edges:
            if not G.has_edge(u, v):
                continue
            r = _judge_one(G, u, v)
            if r["decision"] == "keep":
                keep += 1
            else:
                remove += 1
            confs.append(float(r["confidence"]))
            if keep_rows:
                rows.append(r)

        n = keep + remove
        return {
            "name": name,
            "n": n,
            "keep_rate": keep / max(1, n),
            "remove_rate": remove / max(1, n),
            "avg_confidence": float(np.mean(confs)) if confs else None,
            "rows": rows if keep_rows else None,
        }

    return {
        "diff_sizes": {
            "saber_only_total": len(saber_only_all),
            "base_only_total": len(base_only_all),
            "intersection": len(E_s & E_b),
        },
        "samples": {"sample_each": sample_each, "seed": seed},
        "saber_only_eval": _run_group("saber_only", saber_only, G_saber),
        "base_only_eval": _run_group("base_only", base_only, G_base),
    }


# ============================================================
# Main evaluation runner (complete)
# ============================================================
def evaluate_causal_graphs_v2(
    G_saber: nx.DiGraph,
    G_base: nx.DiGraph,
    *,
    emb: Optional[Dict[str, Any]] = None,
    llm: Optional[Any] = None,
    llm_sample_each: int = 200,
    seed: int = 0,
) -> Dict[str, Any]:
    out: Dict[str, Any] = {"seed": seed}

    # Structural metrics
    out["saber_tr"] = transitive_reduction_metrics(G_saber)
    out["base_tr"] = transitive_reduction_metrics(G_base)

    # Embedding metrics (fairer coherence)
    if emb is not None:
        out["saber_edge_coherence_matched"] = edge_coherence_matched_negatives(
            G_saber, emb, max_pos=5000, seed=seed, per_type=True
        )
        out["base_edge_coherence_matched"] = edge_coherence_matched_negatives(
            G_base, emb, max_pos=5000, seed=seed, per_type=True
        )
        out["path_smoothness_shared"] = path_smoothness_shared_paths(
            G_saber, G_base, emb, path_len_edges=3, num_paths=2000, seed=seed
        )

    # LLM diff preference (BUGFIXED)
    if llm is not None:
        out["llm_preference"] = llm_preference_on_edge_diffs(
            llm,
            G_saber,
            G_base,
            sample_each=llm_sample_each,
            seed=seed,
            keep_rows=True,
        )

    return out


# ============================================================
# Example usage
# ============================================================
# metrics_v2 = evaluate_causal_graphs_v2(
#     G_saber,
#     G_base,
#     emb=emb,
#     llm=llm,
#     llm_sample_each=200,
#     seed=0,
# )
# print(json.dumps(metrics_v2, ensure_ascii=False, indent=2))


In [8]:
emb, emb_stats = build_embedding_map_from_episodes(episodes)  # you already have this
metrics_v2 = evaluate_causal_graphs_v2(
    G_saber,
    G_base,
    emb=emb,
    llm=llm,              # optional
    llm_sample_each=200,
    seed=0,
)
print(json.dumps(metrics_v2, ensure_ascii=False, indent=2))

{
  "seed": 0,
  "saber_tr": {
    "is_dag": true,
    "nodes": 151,
    "edges": 193,
    "edges_tr": 150,
    "reduction_rate": 0.2227979274611399,
    "shortcut_edges": 43,
    "shortcut_ratio": 0.22279792746113988
  },
  "base_tr": {
    "is_dag": true,
    "nodes": 151,
    "edges": 198,
    "edges_tr": 151,
    "reduction_rate": 0.23737373737373735,
    "shortcut_edges": 47,
    "shortcut_ratio": 0.23737373737373738
  },
  "saber_edge_coherence_matched": {
    "pos_n": 193,
    "neg_n": 193,
    "pairs_used": 193,
    "pos_mean": 0.6509336153859744,
    "pos_median": 0.6432631015777588,
    "neg_mean": 0.5353753643082596,
    "neg_median": 0.5367997884750366,
    "gap_mean": 0.11555825107771478,
    "gap_median": 0.10646331310272217,
    "sampling": {
      "strategy": "matched_by_source_u",
      "max_pos": 5000,
      "seed": 0
    },
    "per_type": {
      "precedes": {
        "n": 135,
        "pos_mean": 0.6464620342126222,
        "neg_mean": 0.5403832839038597,
        "